# IA26_P01_G11 — Agente de Routing para Distribuição de Produtos Locais

**Unidade Curricular:** Inteligência Artificial (2026-27) — IPCA, Escola Superior de Tecnologia

---
## 1. Introdução

### Contexto
A procura de produtos alimentares locais está a crescer na UE, mas os pequenos produtores enfrentam custos logísticos elevados e volatilidade de preços. Cadeias de abastecimento curtas reduzem as emissões de transporte e o desperdício alimentar.

O projeto considera uma plataforma digital que liga três tipos de atores:
- **Produtores** — pequenas explorações agrícolas que vendem fruta, legumes e outros produtos;
- **Clientes** — fazem encomendas e escolhem o local de entrega;
- **Centros de recolha (armazéns)** — agregam os produtos antes da distribuição e têm uma frota de veículos.

### Objetivo
Desenvolver um **agente inteligente de routing** que seleciona rotas eficientes de recolha e entrega (*pickup-and-delivery*), respeitando:
- as **janelas de disponibilidade** dos produtos nos produtores;
- as **janelas de entrega** nos locais escolhidos pelos clientes;
- a **capacidade** dos veículos.

Serão aplicados três paradigmas de resolução de problemas:
1. **Pesquisa informada** — Greedy e A*, com heurísticas de distância/tempo;
2. **Pesquisa local** — Hill-Climbing para melhorar uma solução inicial;
3. **CSP** — modelação das janelas temporais e capacidades como restrições.

### Equipa
| Nome | Número |
|------|--------|
| David Pinheiro | 25444 |
| Gonçalo Pinto  | 27999 |
| Ricardo Silva  | 12287 |

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Permitir importar o código da pasta src/ (o notebook está em notebooks/)
sys.path.append(str(Path("..").resolve()))
from src.data_loader import load_dataset

pd.set_option("display.max_columns", None)

---
## 2. Carregamento dos dados

O dataset é sintético e composto por ficheiros CSV que cobrem o distrito de Braga e a Póvoa de Varzim. Existem duas versões: **tiny** (pequena) e **large** (maior), em `data/tiny/` e `data/large/`.

O carregamento está no módulo `src/data_loader.py`, que:
- lê todos os CSV para DataFrames do *pandas*;
- converte as colunas das janelas temporais para `datetime`;
- lê `rules.csv` para um dicionário com os parâmetros do problema;
- cria dicionários de consulta rápida para a matriz de distâncias, usados pelos algoritmos de pesquisa.

In [2]:
# Escolher a versão do dataset: "tiny" ou "large"
DATASET = "tiny"
ds = load_dataset(Path("..") / "data" / DATASET)

# Resumo das tabelas carregadas
pd.DataFrame(
    [(nome, df.shape[0], df.shape[1], ", ".join(df.columns)) for nome, df in ds.tables().items()],
    columns=["tabela", "linhas", "colunas", "atributos"],
)

,tabela,linhas,colunas,atributos
0,localities,49,6,"locality_id, name, municipality, district, lat..."
1,producers,3,6,"producer_id, name, locality_id, address, phone..."
2,products,15,6,"product_id, producer_id, name, category, unit,..."
3,collection_centers,1,5,"center_id, name, locality_id, address, capacit..."
4,vehicles,2,6,"vehicle_id, plate, center_id, capacity_kg, ref..."
5,customers,12,5,"customer_id, name, locality_id, phone, email"
6,delivery_locations,5,5,"delivery_location_id, name, locality_id, addre..."
7,orders,12,5,"order_id, customer_id, delivery_location_id, o..."
8,order_items,40,9,"order_item_id, order_id, product_id, quantity,..."
9,locations,9,7,"location_id, location_type, ref_id, name, muni..."


### 2.1 Regras do problema
O ficheiro `rules.csv` define os parâmetros operacionais. Os mais importantes para o routing são:
- **flow** — todos os produtos passam pelo armazém: produtor → armazém → local de entrega;
- **stop_service_min** — tempo de serviço em cada paragem de recolha/entrega;
- **warehouse_handling_min** — tempo para descarregar/carregar um veículo no armazém;
- **max_warehouse_dwell_h** — tempo máximo que um produto pode ficar no armazém;
- **window_semantics** — as janelas restringem o **início** do serviço.

In [3]:
pd.Series(ds.rules, name="valor").to_frame()

,valor
start_date,2026-10-30
horizon_days,2
flow,producer>warehouse>delivery_location
stop_service_min,10
warehouse_handling_min,15
max_warehouse_dwell_h,24
window_semantics,service_start
avg_speed_kmh,45.0
route_factor,1.3
generator_seed,1


### 2.2 Pré-visualização das tabelas

In [4]:
for nome, df in ds.tables().items():
    print(f"\n=== {nome} ({len(df)} linhas) ===")
    display(df.head())


=== localities (49 linhas) ===


,locality_id,name,municipality,district,latitude,longitude
0,1,Se,Braga,Braga,41.533518,-8.405528
1,2,Maximinos,Braga,Braga,41.564989,-8.435147
2,3,Real,Braga,Braga,41.551572,-8.425425
3,4,Nogueiro,Braga,Braga,41.559380,-8.408464
4,5,Dume,Braga,Braga,41.531493,-8.446483



=== producers (3 linhas) ===


,producer_id,name,locality_id,address,phone,email
0,1,Hortas Valente,24,"Largo Amaral, S/N",(351) 928 063 608,veramatos@oliveira.net
1,2,Campos Leite,40,"Praça Nuno Alves, 28",(351) 924 158 683,noa97@pinho.com
2,3,Horta Morais,36,"Alameda do Ouro, 6",+351961781286,debora07@goncalves.com



=== products (15 linhas) ===


...

,order_item_id,order_id,product_id,quantity,weight_kg,availability_start,availability_end,delivery_start,delivery_end
0,1,1,10,6.3,6.3,2026-10-31 07:30:00,2026-10-31 10:30:00,2026-10-31 10:30:00,2026-10-31 12:45:00
1,2,1,3,3.7,3.7,2026-10-31 08:15:00,2026-10-31 11:15:00,2026-10-31 10:30:00,2026-10-31 12:45:00
2,3,1,15,4.9,4.9,2026-10-31 07:30:00,2026-10-31 10:15:00,2026-10-31 10:30:00,2026-10-31 12:45:00
3,4,1,13,7.1,7.1,2026-10-31 07:15:00,2026-10-31 10:45:00,2026-10-31 10:30:00,2026-10-31 12:45:00
4,5,2,3,1.8,1.8,2026-10-30 07:15:00,2026-10-30 11:15:00,2026-10-31 07:15:00,2026-10-31 10:15:00



=== locations (9 linhas) ===


,location_id,location_type,ref_id,name,municipality,latitude,longitude
0,1,producer,1,Hortas Valente,Fafe,41.474322,-8.174295
1,2,producer,2,Campos Leite,Vieira do Minho,41.633412,-8.125896
2,3,producer,3,Horta Morais,Terras de Bouro,41.716185,-8.340221
3,4,collection_center,1,Centro de Recolha 1,Braga,41.531493,-8.446483
4,5,delivery_location,1,Loja 1,Guimaraes,41.464457,-8.297740



=== distance_matrix (72 linhas) ===


,origin_location_id,origin_type,destination_location_id,destination_type,distance_km,duration_min
0,1,producer,2,producer,23.59,31.4
1,1,producer,3,producer,39.30,52.4
2,1,producer,4,collection_center,30.60,40.8
3,1,producer,5,delivery_location,13.45,17.9
4,1,producer,6,delivery_location,66.45,88.6


### 2.3 Tipos de dados e valores em falta
Confirma que as janelas temporais foram lidas como `datetime` e que não há valores em falta.

In [5]:
for nome, df in ds.tables().items():
    print(f"\n=== {nome} ===")
    display(pd.DataFrame({"tipo": df.dtypes.astype(str), "nulos": df.isna().sum()}))


=== localities ===


,tipo,nulos
locality_id,int64,0
name,str,0
municipality,str,0
district,str,0
latitude,float64,0
longitude,float64,0



=== producers ===


,tipo,nulos
producer_id,int64,0
name,str,0
locality_id,int64,0
address,str,0
phone,str,0
email,str,0



=== products ===


...

,tipo,nulos
order_item_id,int64,0
order_id,int64,0
product_id,int64,0
quantity,float64,0
weight_kg,float64,0
availability_start,datetime64[us],0
availability_end,datetime64[us],0
delivery_start,datetime64[us],0
delivery_end,datetime64[us],0



=== locations ===


,tipo,nulos
location_id,int64,0
location_type,str,0
ref_id,int64,0
name,str,0
municipality,str,0
latitude,float64,0
longitude,float64,0



=== distance_matrix ===


,tipo,nulos
origin_location_id,int64,0
origin_type,str,0
destination_location_id,int64,0
destination_type,str,0
distance_km,float64,0
duration_min,float64,0


### 2.4 Acesso à matriz de distâncias
Os algoritmos trabalham com `location_id` (tabela `locations`), que identifica cada paragem: produtores, centro de recolha e locais de entrega. O `Dataset` disponibiliza dois métodos auxiliares:
- `location_id(tipo, id)` — converte uma entidade no seu `location_id`;
- `distance(origem, destino)` — devolve a distância (km) e o tempo (min) entre duas paragens.

In [6]:
display(ds.locations)

# Exemplo: do produtor 1 ao centro de recolha 1
origem = ds.location_id("producer", 1)
destino = ds.location_id("collection_center", 1)
km, minutos = ds.distance(origem, destino)
print(f"Produtor 1 -> Centro 1: {km} km, {minutos} min")

,location_id,location_type,ref_id,name,municipality,latitude,longitude
0,1,producer,1,Hortas Valente,Fafe,41.474322,-8.174295
1,2,producer,2,Campos Leite,Vieira do Minho,41.633412,-8.125896
2,3,producer,3,Horta Morais,Terras de Bouro,41.716185,-8.340221
3,4,collection_center,1,Centro de Recolha 1,Braga,41.531493,-8.446483
4,5,delivery_location,1,Loja 1,Guimaraes,41.464457,-8.297740
5,6,delivery_location,2,Casa 2,Esposende,41.553679,-8.778977
6,7,delivery_location,3,Casa 3,Guimaraes,41.418772,-8.289729
7,8,delivery_location,4,Casa 4,Vila Nova de Famalicao,41.394239,-8.520320
8,9,delivery_location,5,Loja 5,Guimaraes,41.418772,-8.289729


Produtor 1 -> Centro 1: 30.6 km, 40.8 min
